# Dispersion Backtester — API Demo

Backtest fixed dispersion baskets through the public `backtest()` API — the same
engine the optimizer and the Streamlit app use.

**Covered here:** vol swap · corridor var swap · cross-corridor · long-only ·
missing-data policies · grace days · capped vs uncapped · window control ·
per-leg diagnostics · Excel export.

> Requires a Bloomberg session. Edit the **Inputs** cell of each section and re-run
> top-to-bottom (Kernel → Restart & Run All).

## Setup

Locates the `functions` package (repo root), imports the API, defines display helpers.
If the package is not found, set the `GAIA_REPO` environment variable to your repo root.

In [ ]:
import os, sys
from pathlib import Path

def _find_repo_root() -> Path:
    candidates = [os.environ.get("GAIA_REPO"), os.path.abspath("../.."), os.getcwd(),
                  os.path.abspath(".."), os.path.expanduser("~/Disp")]
    for c in candidates:
        if c and (Path(c) / "functions" / "dispersion").is_dir():
            return Path(c)
    raise FileNotFoundError(
        "Could not locate the 'functions/dispersion' package. "
        "Set GAIA_REPO to your repo root, e.g. os.environ['GAIA_REPO'] = r'C:/path/to/repo'")

sys.path.insert(0, str(_find_repo_root()))

import dataclasses
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import date

from functions.dispersion import backtest, DispersionConfig, get_n_exp_from_date
from functions.dispersion.models import ProductType, MissingDataPolicy
from functions.dispersion._charts import plot_main_backtest  # internal — may change without notice

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


def show_config(config, title="DispersionConfig"):
    """Render every config parameter as a table — all knobs visible at a glance."""
    rows = {f.name: getattr(config, f.name) for f in dataclasses.fields(config)}
    df = pd.DataFrame({"Parameter": rows.keys(), "Value": [str(v) for v in rows.values()]})
    print(f"── {title} " + "─" * max(0, 55 - len(title)))
    display(df.set_index("Parameter"))


def show_basket(df, wcol="Weight (%)", tcol="Variance Asset"):
    """Bar chart of the signed weights."""
    fig = go.Figure(go.Bar(
        x=df[tcol], y=df[wcol],
        marker_color=["#00897b" if w > 0 else "#c62828" for w in df[wcol]]))
    fig.update_layout(title="Basket weights (%)", height=300,
                      margin=dict(t=40, b=30, l=50, r=20))
    fig.show()


def show_result(res, title=""):
    """Metrics summary + cumulative P&L chart for a BacktestResult."""
    if title:
        print(f"── {title} " + "─" * max(0, 60 - len(title)))
    m = res.compute_metrics()
    print(f"hit_ratio {m['hit_ratio']:>8.1f}%   mean {m['mean_return']:>9.4f}   "
          f"last {m['last_value']:>9.2f}   max_dd {res.max_drawdown:>9.2f}   obs {m['n_observations']}")
    cum = res.result_series.cumsum()
    dd = cum - cum.cummax()
    fig = plot_main_backtest(res.timeseries)
    fig.add_trace(go.Scatter(x=dd.index, y=dd, name="Drawdown", yaxis="y2",
                             line=dict(color="#c62828", width=1), fill="tozeroy", opacity=0.3))
    fig.update_layout(height=350, margin=dict(t=40, b=30, l=50, r=20), title=title or None,
                      yaxis2=dict(overlaying="y", side="right", showgrid=False))
    fig.show()


def compare_curves(results: dict, title="Cumulative P&L"):
    """Overlay cumulative curves: {label: BacktestResult}."""
    fig = go.Figure()
    for label, res in results.items():
        fig.add_trace(go.Scatter(x=res.timeseries.index, y=res.result_series.cumsum(),
                                 name=label, line=dict(width=2)))
    fig.update_layout(title=title, height=350, margin=dict(t=40, b=30, l=50, r=20))
    fig.show()

print("Setup OK — repo:", _find_repo_root())

## 1. Inputs

One basket table: **the sign of `Weight (%)` decides the side** (+ = long, − = short).
Longs sum to +100, shorts to −100. (A `Side` column also works if you prefer.)

In [ ]:
# ── Basket (edit me) ──
df_ui = pd.DataFrame([
    {"Underlying": "TSLA US Equity", "Strike (%)": 50.03, "Weight (%)":  15.0},
    {"Underlying": "NVDA US Equity", "Strike (%)": 48.12, "Weight (%)":  15.0},
    {"Underlying": "META US Equity", "Strike (%)": 38.45, "Weight (%)":  15.0},
    {"Underlying": "AMZN US Equity", "Strike (%)": 35.20, "Weight (%)":  15.0},
    {"Underlying": "MSFT US Equity", "Strike (%)": 28.90, "Weight (%)":  15.0},
    {"Underlying": "AAPL US Equity", "Strike (%)": 27.50, "Weight (%)":  15.0},
    {"Underlying": "GOOG US Equity", "Strike (%)": 31.80, "Weight (%)":  10.0},
    {"Underlying": "SPX Index",      "Strike (%)": 20.00, "Weight (%)": -100.0},
])

# ── Dates & tenor (edit me) ──
start_date    = date(2020, 1, 1)
maturity_date = date(2027, 7, 9)

# ── Canonical API columns (UI names → API names) ──
df_vs = df_ui.rename(columns={"Underlying": "Variance Asset",
                              "Strike (%)": "Strike Mono Var Swap (%)"})

tickers = df_ui.loc[df_ui["Weight (%)"] > 0, "Underlying"].tolist()
n_exp = get_n_exp_from_date(maturity_date, tickers)
print(f"Maturity {maturity_date} -> n_exp = {n_exp} trading days")

show_basket(df_vs)
display(df_vs)

## 2. Vol Swap — Long Stocks / Short Index

The base dispersion trade: long single-name vol, short index vol. Every config
parameter is shown explicitly — these are ALL the knobs.

In [ ]:
config_vs = DispersionConfig(
    product_type=ProductType.VOL_SWAP,      # VOL_SWAP or VAR_SWAP_CORRIDOR
    cross_corridor=False,
    n_exp=n_exp,                            # tenor in business days
    local_cap=2.5,                          # per-leg cap: max realized = 2.5 × strike
    is_capped=True,                         # False = uncapped OTC legs
    global_cap=9999999.0,                   # basket-level cap (set e.g. +10 for a note)
    global_floor=-9999999.0,                # basket-level floor (set e.g. -10 for a note)
    missing_data_policy=MissingDataPolicy.ADAPTIVE_REWEIGHT,
    reweight_grace_days=0,                  # gap days a name keeps its last mark
)
show_config(config_vs)

result_vs = backtest(df_vs, config_vs, start_date=start_date)
show_result(result_vs, "Vol Swap — L/S dispersion")

## 3. Corridor Variance Swap

Same basket, but variance only accrues while spot is inside the `[70%, 130%]` corridor.

In [ ]:
config_corr = DispersionConfig(
    product_type=ProductType.VAR_SWAP_CORRIDOR,
    cross_corridor=False,
    n_exp=n_exp,
    barrier_up=1.30,                        # upper corridor (130% spot)
    barrier_down=0.70,                      # lower corridor (70% spot)
    local_cap=2.5,
    is_capped=True,
    missing_data_policy=MissingDataPolicy.ADAPTIVE_REWEIGHT,
)
show_config(config_corr)

result_corr = backtest(df_vs, config_corr, start_date=start_date)
show_result(result_corr, "Corridor Var Swap — L/S dispersion")

## 4. Cross-Corridor

Each leg = mono var swap on the **stock** minus corridor var swap on the **index**
(inside the stock's corridor). Input uses the canonical cross columns.

In [ ]:
df_xc = pd.DataFrame([
    {"Variance Asset": "SPX Index", "Corridor Condition Asset": "TSLA US Equity", "Strike Cross Corridor (%)": 19.24, "Strike Mono Var Swap (%)": 50.03, "Weight (%)":  15.0},
    {"Variance Asset": "SPX Index", "Corridor Condition Asset": "NVDA US Equity", "Strike Cross Corridor (%)": 18.80, "Strike Mono Var Swap (%)": 48.12, "Weight (%)":  15.0},
    {"Variance Asset": "SPX Index", "Corridor Condition Asset": "META US Equity", "Strike Cross Corridor (%)": 20.10, "Strike Mono Var Swap (%)": 38.45, "Weight (%)":  15.0},
    {"Variance Asset": "SPX Index", "Corridor Condition Asset": "AMZN US Equity", "Strike Cross Corridor (%)": 19.55, "Strike Mono Var Swap (%)": 35.20, "Weight (%)":  15.0},
    {"Variance Asset": "SPX Index", "Corridor Condition Asset": "MSFT US Equity", "Strike Cross Corridor (%)": 21.20, "Strike Mono Var Swap (%)": 28.90, "Weight (%)":  15.0},
    {"Variance Asset": "SPX Index", "Corridor Condition Asset": "AAPL US Equity", "Strike Cross Corridor (%)": 21.40, "Strike Mono Var Swap (%)": 27.50, "Weight (%)":  15.0},
    {"Variance Asset": "SPX Index", "Corridor Condition Asset": "SPX Index",   "Strike Cross Corridor (%)": 20.00, "Strike Mono Var Swap (%)": 20.00, "Weight (%)": -100.0},
])

config_xc = DispersionConfig(
    product_type=ProductType.VAR_SWAP_CORRIDOR,
    cross_corridor=True,
    n_exp=n_exp,
    barrier_up=1.30, barrier_down=0.70,
    local_cap=2.5, is_capped=True,
    missing_data_policy=MissingDataPolicy.ADAPTIVE_REWEIGHT,
)
show_config(config_xc)
show_basket(df_xc, tcol="Corridor Condition Asset")

result_xc = backtest(df_xc, config_xc, start_date=start_date)
show_result(result_xc, "Cross-Corridor — L/S dispersion")

# Per-leg mono/cross split is available in cross-corridor mode
if result_xc.cross_leg_pnl is not None:
    display(result_xc.cross_leg_pnl.tail(3))

## 5. Long-Only Basket

No short leg — drop the negative rows; the "Short Leg" column is simply zero.

In [ ]:
df_long_only = df_vs[df_vs["Weight (%)"] > 0].copy()
show_basket(df_long_only)

result_lo = backtest(df_long_only, config_vs, start_date=start_date)
show_result(result_lo, "Vol Swap — long-only")

## 6. Missing-Data Policies

How to treat days where a leg has no print:

| Policy | Behavior |
|--------|----------|
| `ADAPTIVE_REWEIGHT` | weights redistribute over the legs that **did** trade (basket stays fully invested) |
| `FILL_ZERO` | missing leg contributes 0 that day (basket under-invested) |
| `DROP_INCOMPLETE_DAYS` | day removed entirely unless **every** weighted leg trades |

In [ ]:
results_pol = {}
for pol in [MissingDataPolicy.ADAPTIVE_REWEIGHT,
            MissingDataPolicy.FILL_ZERO,
            MissingDataPolicy.DROP_INCOMPLETE_DAYS]:
    cfg = DispersionConfig(
        product_type=ProductType.VOL_SWAP, cross_corridor=False, n_exp=n_exp,
        local_cap=2.5, missing_data_policy=pol)
    results_pol[pol.name] = backtest(df_vs, cfg, start_date=start_date)

comp = pd.DataFrame({
    name: r.compute_metrics() for name, r in results_pol.items()
}).T[["hit_ratio", "mean_return", "last_value", "n_observations"]]
display(comp.round(3))
compare_curves(results_pol, "Missing-data policies — cumulative P&L")

## 7. Grace Days (`reweight_grace_days`)

Under `ADAPTIVE_REWEIGHT`, a gapped name keeps its weight (carrying its **last mark**)
for up to N days before the basket redistributes — avoids recomposing on one-day hiccups.

In [ ]:
config_grace = DispersionConfig(
    product_type=ProductType.VOL_SWAP, cross_corridor=False, n_exp=n_exp,
    local_cap=2.5,
    missing_data_policy=MissingDataPolicy.ADAPTIVE_REWEIGHT,
    reweight_grace_days=3,          # 0 = redistribute immediately
)
show_config(config_grace)

result_grace = backtest(df_vs, config_grace, start_date=start_date)
compare_curves({"grace=0": result_vs, "grace=3": result_grace},
               "Adaptive reweight — grace days effect")

## 8. Capped vs Uncapped Legs (`is_capped`)

`is_capped=True` caps each leg's realized variance at `local_cap × strike` (note-style
legs). `False` = uncapped OTC legs. (Independent of the **global** basket cap/floor,
which stays unlimited here.)

In [ ]:
config_uncapped = DispersionConfig(
    product_type=ProductType.VOL_SWAP, cross_corridor=False, n_exp=n_exp,
    local_cap=2.5, is_capped=False,        # uncapped OTC legs
    missing_data_policy=MissingDataPolicy.ADAPTIVE_REWEIGHT,
)
show_config(config_uncapped)

result_uncapped = backtest(df_vs, config_uncapped, start_date=start_date)
compare_curves({"capped (local_cap=2.5)": result_vs, "uncapped OTC": result_uncapped},
               "Capped vs uncapped legs — cumulative P&L")

## 9. Window Control & Per-Leg Diagnostics

- `start_date` / `end_date`: the delivered curve is clipped to `[start_date, end_date]`
  (kernels still get full warm-up before `start_date`).
- `per_leg_pnl`: daily P&L of each leg · `active_legs_count`: invested names per day.

In [ ]:
result_win = backtest(df_vs, config_vs,
                      start_date=date(2022, 1, 1), end_date=date(2023, 12, 31))
show_result(result_win, "Windowed backtest 2022–2023")

# Per-leg cumulative contributions
fig = go.Figure()
for col in result_win.per_leg_pnl.columns:
    fig.add_trace(go.Scatter(x=result_win.per_leg_pnl.index,
                             y=result_win.per_leg_pnl[col].cumsum(),
                             name=col, line=dict(width=1.5)))
fig.update_layout(title="Per-leg cumulative P&L", height=380,
                  margin=dict(t=40, b=30, l=50, r=20))
fig.show()

if result_win.active_legs_count is not None:
    print("Active legs:", result_win.active_legs_count.value_counts().sort_index().to_dict())

## 10. Export

`to_frames()` returns every table as DataFrames; write them to **one Excel workbook**
(no zip needed).

In [ ]:
frames = result_vs.to_frames()
print("Tables:", list(frames))

with pd.ExcelWriter("backtest_export.xlsx", engine="xlsxwriter") as writer:
    for name, df in frames.items():
        df.to_excel(writer, sheet_name=str(name)[:31], index=True)
print("Written: backtest_export.xlsx")

---
## Conventions

| Topic | Rule |
|-------|------|
| Weight sign | **+ = long, − = short** (a `Side` column also works) |
| Result | `Result = Long Leg + Short Leg` — Short Leg is stored **negated** (negative on high-index-vol days) |
| Strikes | input in **percent** (`50.03` = 50.03% vol) |
| `max_drawdown` | equity-curve peak-to-trough in **raw P&L units**, ≤ 0 (same as per-stock stats) |
| Warm-up | rolling kernels see full history before `start_date` — row 0 is a valid observation |
| Optimizer alignment | the optimizer scores this exact curve; backtesting the winning basket with the same config reproduces it |